In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

from xgboost import XGBClassifier

In [6]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [7]:
train_df["Heart Disease"] = train_df["Heart Disease"].map({
    "Presence": 1,
    "Absence": 0
})

In [8]:
X = train_df.drop(["id", "Heart Disease"], axis=1)
y = train_df["Heart Disease"]

test_ids = test_df["id"]
X_test_final = test_df.drop("id", axis=1)

In [9]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test_final_scaled = scaler.transform(X_test_final)

In [11]:
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [12]:
y_pred = model.predict(X_valid)
y_proba = model.predict_proba(X_valid)[:, 1]

print("Accuracy:", accuracy_score(y_valid, y_pred))
print("ROC-AUC:", roc_auc_score(y_valid, y_proba))
print(classification_report(y_valid, y_pred))

Accuracy: 0.8898253968253969
ROC-AUC: 0.9560418519987457
              precision    recall  f1-score   support

           0       0.89      0.91      0.90     69509
           1       0.88      0.87      0.88     56491

    accuracy                           0.89    126000
   macro avg       0.89      0.89      0.89    126000
weighted avg       0.89      0.89      0.89    126000



In [13]:
# Scale full training data
X_scaled_full = scaler.fit_transform(X)

# Train on full dataset
model.fit(X_scaled_full, y)

# Scale test data again using full scaler
X_test_final_scaled = scaler.transform(X_test_final)

# Predict test
final_test_pred = model.predict(X_test_final_scaled)

In [16]:
import pickle
pickle.dump(model,open('model.pkl','wb'))